# M2 — SQL + Python

> 從零開始，學會用 Python 操作資料庫

## 學習目標

| 你會學到 | 為什麼重要 |
|:---------|:-----------|
| `sqlite3` 連線、游標、執行 SQL | 所有 DB 操作的底層都是這套流程 |
| 執行 `.sql` 腳本檔 | 把建表邏輯和 Python code 分開 |
| `pd.read_sql()` | 一行把查詢結果變 DataFrame |
| 參數化查詢（`?`） | 防 SQL injection 的鐵律 |
| `df.to_sql()` | 把 DataFrame 寫回資料庫 |
| Context manager | 自動 commit、自動關連線 |

## 為什麼用 SQLite？

- **零安裝**：Python 內建 `sqlite3`，不用裝 MySQL/PostgreSQL
- **單檔存檔**：整個資料庫就是一個 `.db` 檔，方便搬移
- **完整 SQL**：該有的 JOIN、GROUP BY、子查詢、Transaction 都有
- **真的在用**：Android、iOS、瀏覽器、很多 embedded 系統都用 SQLite

學會 SQLite 之後，換 PostgreSQL/MySQL 只是換連線字串的事。


---

## 1. sqlite3 基本操作

每次跟資料庫互動都是這四步：

```
1. connect    → 拿到 connection
2. cursor     → 從 connection 拿到 cursor
3. execute    → 用 cursor 執行 SQL
4. commit     → 寫入操作要 commit 才生效
(5. close)    → 用完關掉
```

查詢不用 commit，但寫入（INSERT/UPDATE/DELETE）一定要。


In [ ]:
import sqlite3
from pathlib import Path

# 連線（檔案不存在會自動建立）
DB_PATH = Path("school.db")
conn = sqlite3.connect(DB_PATH)

# 拿 cursor 來執行 SQL
cursor = conn.cursor()

# 執行一行 SQL
cursor.execute("SELECT sqlite_version()")
print("SQLite 版本：", cursor.fetchone())

conn.close()

---

## 2. 執行 SQL 腳本檔

正式專案通常會把建表的 SQL 寫在獨立的 `.sql` 檔，用 Python 載入執行。

好處：
- DBA 可以用 DB 工具直接編輯、review
- Git diff 看建表變更很清楚
- 同一份 SQL 可以用在 Python、shell、其他語言

`cursor.executescript()` 可以一次執行多個 statement。


In [ ]:
def run_sql_file(conn: sqlite3.Connection, sql_path: Path) -> None:
    """讀取 .sql 檔並執行裡面所有的 statement"""
    sql = sql_path.read_text(encoding="utf-8")
    conn.executescript(sql)
    conn.commit()

# 從頭建一個乾淨的資料庫
if DB_PATH.exists():
    DB_PATH.unlink()    # 刪掉舊的

conn = sqlite3.connect(DB_PATH)

run_sql_file(conn, Path("sql/02_create_tables.sql"))
run_sql_file(conn, Path("sql/03_insert_data.sql"))

print("建表 + 塞資料完成")
conn.close()

---

## 3. 純 sqlite3 查詢

先用原生 sqlite3 查一次，等等再用 Pandas 對比。

- `fetchone()` → 拿一筆
- `fetchmany(n)` → 拿 n 筆
- `fetchall()` → 全部拿


In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT sId, sName, sGender FROM students")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

注意 `cursor.fetchall()` 回傳的是 tuple list，沒有欄位名稱，用起來不方便。

**解法：** 用 `pandas.read_sql()`，直接拿到 DataFrame ──


---

## 4. Pandas 的 `read_sql()` — 這就是 SQL + Python 的殺手鐧

```python
df = pd.read_sql("SELECT * FROM students", conn)
```

一行，就把查詢結果變成 DataFrame。欄位名稱、型別全自動處理。

In [ ]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)

students = pd.read_sql("SELECT * FROM students", conn)
students

In [ ]:
# 有欄位名稱、可以用 .describe() / .info() / .groupby() 等全部 Pandas 功能
students.info()

### JOIN 查詢：學生成績報表

這就是 SQL 最經典的用法 — 跨多張表 JOIN 拉出報表。


In [ ]:
query = """
SELECT
    s.sName   AS 學生,
    c.cName   AS 課程,
    sc.score  AS 分數,
    t.tName   AS 授課老師
FROM scores sc
JOIN students s ON sc.sId = s.sId
JOIN courses  c ON sc.cId = c.cId
JOIN teachers t ON c.tId  = t.tId
ORDER BY sc.score DESC
"""

report = pd.read_sql(query, conn)
report

---

## 5. 參數化查詢（CRITICAL — 安全！）

**絕對不要用字串拼接組 SQL：**

```python
# ❌❌❌ SQL Injection 大門敞開
student_id = input("輸入學號：")
cursor.execute(f"SELECT * FROM students WHERE sId = '{student_id}'")
```

如果使用者輸入 `' OR '1'='1`，整張表的資料就被撈光了。

**正確做法：用 `?` 當 placeholder，把值當參數傳進去：**


In [ ]:
# ✅ 安全的寫法
student_id = "087"

df = pd.read_sql(
    "SELECT * FROM students WHERE sId = ?",
    conn,
    params=(student_id,),    # 用 tuple 傳參數
)
df

In [ ]:
# 多個參數也一樣
min_score = 80
gender = "女"

df = pd.read_sql(
    """
    SELECT s.sName, c.cName, sc.score
    FROM scores sc
    JOIN students s ON sc.sId = s.sId
    JOIN courses  c ON sc.cId = c.cId
    WHERE sc.score >= ? AND s.sGender = ?
    """,
    conn,
    params=(min_score, gender),
)
df

---

## 6. `df.to_sql()` — 把 DataFrame 寫回資料庫

ETL 的「L」(Load) 就是這一步。

常見情境：
- 把 CSV 清理乾淨後，存進 DB 當 raw_data
- Pandas 算完統計後，把結果表寫回 DB 給 BI 工具用


In [ ]:
# 假設我們有一份計算過的報表
summary = (
    pd.read_sql("SELECT sId, score FROM scores", conn)
    .groupby("sId", as_index=False)
    .agg(avg_score=("score", "mean"), subjects=("score", "count"))
)
summary

In [ ]:
# 寫回 DB
summary.to_sql(
    name="student_summary",    # 新表名稱
    con=conn,
    if_exists="replace",       # 'fail'/'replace'/'append'
    index=False,
)
conn.commit()

# 驗證：從 DB 重新讀回來
pd.read_sql("SELECT * FROM student_summary", conn)

### `if_exists` 三種模式

| 值 | 行為 | 使用時機 |
|:---|:-----|:---------|
| `'fail'` | 表已存在就拋錯（預設） | 不想覆蓋資料時 |
| `'replace'` | 把舊表 drop 掉重建 | 每次完整覆蓋（如每日統計表） |
| `'append'` | 繼續塞進舊表 | Log 類資料、時間序列累積 |


---

## 7. Context Manager — 自動 commit 和關連線

手動寫 `conn.close()` 很容易忘，忘了會有資源洩漏。用 `with` 自動處理：


In [ ]:
# ✅ 推薦寫法：with block 結束時自動 commit，異常時自動 rollback
with sqlite3.connect(DB_PATH) as conn:
    conn.execute(
        "UPDATE students SET sNickname = ? WHERE sId = ?",
        ("超人", "087"),
    )
    # with 結束時自動 commit

# 驗證
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("SELECT * FROM students WHERE sId = '087'", conn)
df

> **注意：** `sqlite3` 的 `with` 只會 auto-commit，**不會自動 close 連線**。
> 如果你在函式裡用，通常還是自己 `conn.close()` 一下。
> 或者用 `contextlib.closing()` 包起來。


---

## 8. Transaction 與 rollback

一筆交易（如轉帳）要嘛全部成功、要嘛全部回復。這就是 Transaction。

sqlite3 預設 auto-commit 是關的，所以每次 `execute` 後要 `commit` 才真的寫進去。
如果中途出錯，呼叫 `rollback()` 可以把這次的改動全部取消。


In [ ]:
conn = sqlite3.connect(DB_PATH)
try:
    conn.execute("INSERT INTO students VALUES ('999', '測試生', '男', '小測')")
    conn.execute("INSERT INTO students VALUES ('003', '重複', '男', '壞壞')")  # PRIMARY KEY 衝突
    conn.commit()
    print("全部成功")
except sqlite3.IntegrityError as e:
    conn.rollback()
    print(f"失敗，已 rollback：{e}")
finally:
    conn.close()

# 驗證：999 這筆應該也沒存進去
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("SELECT * FROM students WHERE sId IN ('999', '003')", conn)
df

---

## 9. 練習

把下面的題目做一遍，你就把 M2 吸收了八成：

### 🟢 基礎

1. 查詢所有**女學生**的姓名和暱稱
2. 查詢**程式設計**這門課的**平均分數**
3. 把一筆新的老師資料 `('T099', '新老師')` 寫進 teachers 表

### 🟡 進階

4. 用一個 JOIN 查詢列出：**每位學生選了幾門課、總學分**
5. 找出**平均成績最高的前 3 名學生**（姓名 + 平均分）
6. 把第 5 題的結果用 `to_sql` 寫成 `top3_students` 新表

### 🔴 挑戰

7. 寫一個函式 `update_score(sId, cId, new_score)`，要求：
   - 用參數化查詢（防 injection）
   - 包在 try/except 裡，失敗自動 rollback
   - 成功回傳 `True`、失敗回傳 `False`


In [ ]:
# 🟢 1. 查詢所有女學生
# 你的程式碼：


In [ ]:
# 🟡 4. 每位學生選了幾門課、總學分
# 你的程式碼：


In [ ]:
# 🔴 7. update_score 函式
# 你的程式碼：


---

## 小結

1. **sqlite3** 是 Python 內建的 DB 模組，零安裝
2. **四步驟**：connect → cursor → execute → commit
3. **`pd.read_sql()`** 是 Pandas + SQL 的黃金組合
4. **參數化查詢**（`?` + `params=(...)`) 是安全鐵律
5. **`df.to_sql()`** 把 DataFrame 寫回 DB，搭配 `if_exists` 控制覆蓋策略
6. **Transaction**：失敗 `rollback`、成功 `commit`，一致性的保證

下一步：M3 學會怎麼記錄 pipeline 的執行狀況。